# B06 — Volume index grating

## Mathematical and physical model

The structure is nonmagnetic and periodic in the transverse plane. In every constant-$z$ slice the electric field satisfies
$$
\nabla\times\nabla\times\mathbf E-k_0^2\epsilon_r(\mathbf r)\mathbf E=0,\qquad
\mathbf E_t(\mathbf r)=\sum_{m,n}\mathbf e_{mn}(z)e^{i(\mathbf k_{\parallel,0}+\mathbf G_{mn})\cdot (x,y)},\qquad
\mathbf G_{mn}=2\pi(m/P_x,n/P_y).
$$
Match tangential $\mathbf E$ and $\mathbf H$ across the layer interfaces. In each exterior medium $j$, the outgoing order is propagating when
$k_{z,j,mn}^2=k_0^2\epsilon_j-|\mathbf k_{\parallel,0}+\mathbf G_{mn}|^2>0$.
The comparison uses the complex amplitudes normalized so $|a_{mn}^{q\leftarrow p}|^2$ is the incident-power fraction in an outgoing polarization channel. A one-dimensional grating retains $n=0$ even when the incident $k_y$ is nonzero.

The index is $n(x)=n_0+\Delta n\cos(2\pi x/P_x)$ and $\epsilon_r(x)=n(x)^2$, with the surrounding index set to $n_0$. The approximate Bragg direction follows by making the zeroth and $-1$ order longitudinal wavevector magnitudes equal:
$$
\sin\theta_B=\frac{\lambda_0}{2n_0P_x}.
$$
For the specified parameters this lies near $12.2^\circ$. The numerical model approximates the continuous modulation by index strips whose width is set by the raster.

## Physical scenario

A 10 µm thick film contains a weak sinusoidal index modulation across the grating direction. The neighboring media match its mean index. Varying incidence angle through the Bragg condition transfers energy from the incident beam to a selected transmitted order within a narrow angular band.

## Possible applications

This is the mechanism of angle-selective volume holograms and Bragg couplers, including waveguide couplers for display optics. [Ansys volume holographic grating example](https://optics.ansys.com/hc/en-us/articles/37118679173267-Volume-Holographic-Grating)

## Benchmark objective

Compare the Bragg angular response of a thick sinusoidal index grating in index-matched surroundings.

## Reproduce

Use the project environment and select **Run All**. The notebook and CLI use [case.yaml](case.yaml). The native [Lumerical model](model.fsp) and [reconstruction script](model.lsf) correspond to quick-sweep point 0.

[Published comparisons and figures](../../../BENCHMARK_REPORT.md#b06) · [Periodic diagnosis](../../../PERIODIC_DIAGNOSIS.md#b06)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("B06")[0]
samples, settings, metadata = case_setup(case_path)
results = {}

print(f"{len(samples)} points, Fourier truncation index {settings.order}")
figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe, solve

print(probe())
results["meent"] = solve_sweep(solve, samples, settings)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe, solve

print(probe())
results["torcwa"] = solve_sweep(solve, samples, settings)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe, solve

print(probe())
results["grcwa"] = solve_sweep(solve, samples, settings)

## Lumerical RCWA

This cell uses the installed, licensed FDTD application through its Python API. It reuses one session
across the sweep and evaluates the same model generator used by the reconstruction script.

In [ ]:
from rcwa_cv.solvers.lumerical import session, solve

with session() as fdtd:
    print(f"Lumerical build {fdtd.version()}")
    results["lumerical"] = solve_sweep(solve, samples, settings)

## Compare with Lumerical

The spectra show total reflection, transmission and the co-polarized transmission phase.
The order plot uses a shared excitation point. The table and difference plots compare each open-source solver with Lumerical.
Periodic quick sweeps provide preliminary comparisons.

In [ ]:
show_versions(results)
show_results(metadata, results)